# PrecisionSkin – Hyperpigmentation Auto-Box Tool (pseudo-labelling)

Trains a **hyperpigmentation-only** box detector (one class, `hyperpigmentation`, class ID 0) on your annotated photos,
then runs it on your photos **without** boxes and writes **pseudo-labels** (YOLO format) in a separate folder.

**Run the cells in order (Run All works).** Cell 7 (training) is the long one; everything else takes minutes.

| Cell | What it does |
|---|---|
| 1 | Checks the packages (installs nothing unless you ask) |
| 2 | **Settings** – the only cell you may edit (folders, thresholds) |
| 3 | GPU check |
| 4 | Builds the one-class training set. Validation (164 photos / 448 boxes) and test (164 / 380) stay the fixed v1 sets |
| 5 | Checks the unannotated photos: unreadable, duplicates, copies of val/test photos, copies of training photos |
| 6 | Box audit: do the public boxes follow the same rule as your hand boxes? |
| 7 | Trains the variants n_640, s_800, m_800 (resumable) |
| 8 | **Comparison** v1 vs the new models on v1 val, v1 test and your hand-boxed test set |
| 9 | Choose the model for annotation |
| 10 | Trial run on 100 photos – **look at the previews before the full run** |
| 11 | Full batch annotation with resume (stop and re-run any time) |
| 12 | Summary table + a contact sheet of previews |
| 13 | Review folder for the uncertain / no-detection photos (optional) |

**Never touched:** your original photos, the v1 dataset, your hand-boxed folder. Everything generated goes to `OUT_ROOT/RUN_NAME/`.

**Read this:** *no detection does not mean healthy skin.* Photos where the model finds nothing are only flagged `no_detection`.
Pseudo-labels are the model's guess; boxes it missed stay missing. Do not use them as proof of "no lesion",
and do not mix them into the test set. The v2 eczema experiment showed that training on boxes with missed lesions made the model worse.

## CELL 1 – Setup and dependencies

In [ ]:
# CELL 1 – Setup and dependencies
# One-time install inside the detection environment (WSL Ubuntu: ~/PrecisionSkinV1/.venv_det):
#     pip install ultralytics opencv-python-headless pandas pyyaml numpy matplotlib
# (torch with CUDA is already in .venv_det). Set INSTALL_MISSING = True to let this cell do it.
import importlib, subprocess, sys
INSTALL_MISSING = False
need = {"ultralytics": "ultralytics", "cv2": "opencv-python-headless", "pandas": "pandas", "yaml": "pyyaml", "numpy": "numpy", "matplotlib": "matplotlib"}
missing = [pip for mod, pip in need.items() if importlib.util.find_spec(mod) is None]
if missing and INSTALL_MISSING:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing]); missing = []
print("missing packages:", missing or "none")
assert not missing, "Install: pip install " + " ".join(missing) + "   (or set INSTALL_MISSING = True and run again)"

# hyperpig_autolabel.py must sit next to this notebook (or in the SkinDevApp folder)
from pathlib import Path
for cand in (Path.cwd(), Path("/mnt/c/Users/Bautista/Downloads"), Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp")):
    if (cand / "hyperpig_autolabel.py").exists():
        sys.path.insert(0, str(cand)); print("using", cand / "hyperpig_autolabel.py"); break
else:
    raise FileNotFoundError("hyperpig_autolabel.py not found - copy it next to this notebook")
import hyperpig_autolabel as H

## CELL 2 – Settings (edit only this cell)

In [ ]:
# CELL 2 – Settings
from pathlib import Path
HOME = Path.home() / "PrecisionSkinV1"
cfg = H.default_config()
cfg.update(
    # ---- inputs (read only) ----
    v1_dataset = HOME / "detector_work" / "dataset_v1",                                           # v1 train/val/test (+ dataset_manifest.csv)
    hand_boxed_dir = Path("/mnt/c/Users/Bautista/Downloads/own_hyperpigmentation.yolov8"),         # your hand boxes (images/ + labels/, class 0)
    unlabeled_dir = Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Classes/Hyperpigmentation"), # the ~3,000 photos WITHOUT boxes (searched recursively)
    # ---- outputs ----
    out_root = HOME / "hyperpig_autolabel",
    run_name = "hyperpig_only_v1",
    # ---- training ----
    epochs = 100, patience = 40, workers = 2, seed = 42,
    # model variants (one class each). Bigger model / higher resolution need a smaller batch on the 8 GB GPU.
    variants = [dict(name="n_640", model="yolov8n.pt", imgsz=640, batch=16),
                dict(name="s_800", model="yolov8s.pt", imgsz=800, batch=8),
                dict(name="m_800", model="yolov8m.pt", imgsz=800, batch=4)],      # remove a line to skip a variant (m_800 takes the longest)
    active_variant = "s_800",     # used for the auto-annotation. Change it after you see the comparison (cell 8)
    # ---- consistency of the boxes ----
    hand_holdout = 100,           # of your hand-boxed photos, this many are kept OUT of training as a hand-boxed test set
    public_max_box_area = None,   # after cell 5: set e.g. 0.5 to drop public training photos that contain a box over 50% of the photo
    # ---- thresholds for the pseudo-labels ----
    conf_min = 0.15,      # below this: ignored completely
    conf_accept = 0.40,   # >= this: box is written to the pseudo-label file
    conf_high = 0.55,     # an image whose best box is below this is flagged "weak" (uncertain)
    preview_mode = "all", # "all" | "uncertain" | "none"
    review_max = 500,     # how many uncertain / no-detection photos go into the review folder
)
print("outputs ->", cfg.out_root / cfg.run_name)

## CELL 3 – GPU check

In [ ]:
# CELL 3 – GPU check
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (training will be very slow)")

## CELL 4 – Build the one-class training set
Class IDs: v1 labels use `1 = hyperpigmentation`; this cell rewrites them to `0`. Your hand-boxed photos replace the public boxes of the same photo.
The validation and test sets are copied unchanged from v1. If the counts differ from your expected 164/448 and 164/380 you get a warning.

In [ ]:
# CELL 4 – Build the one-class dataset
report = H.build_dataset(cfg)
import pandas as pd
display(pd.DataFrame([{"split": s, "photos": report[s + "_photos"], "boxes": report[s + "_boxes"]} for s in ("train", "val", "test")]))

## CELL 5 – Check the photos without boxes
Each photo is compared (32×32 thumbnail, 8 rotations/flips, correlation ≥ 0.97) with **every** annotated hyperpigmentation photo.
Only photos with status `ok` are annotated. Others are listed in `reports/overlap_report.csv`:
`overlap_with_val`, `overlap_with_test`, `already_annotated`, `duplicate_of_other_unlabeled`, `unreadable`.

In [ ]:
# CELL 5 – Index and overlap check
idx = H.index_unlabeled(cfg)
display(idx.status.value_counts().rename("photos").to_frame())

## CELL 6 – Do the public boxes follow the same rule as your hand boxes?
Compares box size and count per photo. If the public boxes are much larger (whole-face boxes) or the number per photo is very different,
the public boxes teach the model a different rule than yours. In that case set `public_max_box_area = 0.5` in cell 2 and run cells 4-6 again
(photos with a huge public box are then left out of training; nothing is deleted from your folders).

In [ ]:
# CELL 6 – Box audit
audit = H.box_audit(cfg)
display(audit)

## CELL 7 – Train the one-class variants (n_640, s_800, m_800)

Long (hours in total). Each variant is skipped if it is already trained. If the kernel stops, set `RESUME = True` and run again. Variants are trained one after another; you can stop after any variant and continue with Cell 8.

In [ ]:
# CELL 7 – Training (long)
RESUME = False
for v in cfg.variants:
    print("=" * 20, v["name"], v["model"], "imgsz", v["imgsz"], "batch", v["batch"])
    H.train(cfg, variant=v["name"], resume=RESUME)

## CELL 8 – Comparison: v1 (in the app) vs the new one-class models
Same photos, same scoring code for every model: your **v1 validation**, **v1 test** and your **hand-boxed test set** (photos you boxed yourself that were never trained on).
For the new models the confidence used for precision/recall is the best-F1 value on validation; v1 uses the app's 0.25. Read the hand-boxed test rows first: they measure your box rule.
Caution: v1 may have seen the public copy of a hand-boxed test photo during its training, which favours v1 slightly.

In [ ]:
# CELL 8 – Comparison
cmp = H.compare(cfg)
display(cmp.pivot_table(index="model", columns="set", values="mAP@0.5", sort=False).round(3).rename(columns=lambda c: "mAP@0.5 - " + c))
display(cmp)

## CELL 9 – Choose the model for the auto-annotation

Set `cfg.active_variant` to the best model on the hand-boxed test set (and not worse on v1 test), then run this cell. It also gives the standard scores of that model.

In [ ]:
# CELL 9 – Choose the model
cfg.active_variant = "s_800"        # <- change to the best variant from Cell 8
print("active model:", H.weights_path(cfg))
display(cmp[cmp.model == cfg.active_variant])

## CELL 10 – Trial run (100 photos) – look at the previews first

In [ ]:
# CELL 10 – Trial run
H.annotate(cfg, limit=100)
from pathlib import Path
import cv2, matplotlib.pyplot as plt, random
P = H.paths(cfg)
files = sorted(P.previews.rglob("*.jpg")); random.Random(0).shuffle(files)
fig, ax = plt.subplots(3, 4, figsize=(16, 12))
for a, f in zip(ax.ravel(), files[:12]):
    a.imshow(cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)); a.set_title(f.parent.name + " / " + f.stem[-12:], fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()
print("green solid = accepted (written to pseudo-label), orange dashed = uncertain (only in confidences/ and review)")

## CELL 11 – Full batch annotation (resume supported)
Stop it any time (interrupt the kernel); run the cell again and it continues with the photos not yet processed.
To re-do everything with other thresholds, delete the `pseudo_labels` folder inside the run folder first.

In [ ]:
# CELL 11 – Annotate everything
summary = H.annotate(cfg)

## CELL 12 – Summary
Output folders in `OUT_ROOT/RUN_NAME/`:
- `pseudo_labels/labels/*.txt` – YOLO boxes (`0 cx cy w h`), only boxes ≥ `conf_accept`
- `pseudo_labels/confidences/*.txt` – ALL predicted boxes ≥ `conf_min` with confidence and tier (`0 cx cy w h conf accepted|uncertain`)
- `pseudo_labels/images_summary.csv`, `boxes.csv` – per image / per box tables
- `previews/accepted|uncertain|no_detection/` – images with boxes and confidences drawn
- `reports/` – `annotation_summary.json`, `review_queue.csv`, `unlabeled_index.csv`, `overlap_report.csv`, `dataset_report.json`, `model_metrics.json`

In [ ]:
# CELL 12 – Summary
s = H.summarize(cfg)
display(pd.DataFrame([
    ["Images processed", s["images_processed"]], ["  accepted (all boxes >= conf_accept, no warnings)", s["images_accepted"]],
    ["  uncertain (low-confidence boxes or warnings)", s["images_uncertain"]], ["  no detection (NOT necessarily healthy skin)", s["images_no_detection"]],
    ["Boxes accepted (in the pseudo-labels)", s["boxes_accepted"]], ["Boxes uncertain (not in the pseudo-labels)", s["boxes_uncertain"]],
    ["Images usable as pseudo-labelled training data", s["images_usable_for_training"]],
], columns=["", "count"]))
print("warnings per image:", s["images_per_flag"])
print("accepted-box confidence:", s.get("accepted_box_confidence"))

## CELL 13 – Review folder for uncertain / no-detection photos (optional)
Copies the photos in `reports/review_queue.csv` (uncertain first, most confident first) into `review_folder/` with the model's boxes as a starting draft.
Open it with your box tool, fix or add boxes, and save. Only photos you actually fix should be used as manual labels.

In [ ]:
# CELL 13 – Review folder
folder = H.make_review_folder(cfg)
print()
print("Open this folder with lesion_box_tool.py (see ROBOFLOW/box tool usage), --label hyperpigmentation:")
print(folder)

## How to use the result safely
1. Check 50-100 previews by eye (Cell 10/12). If many boxes are wrong, raise `conf_accept` (0.5-0.6) and re-run Cell 11 after deleting `pseudo_labels/`.
2. Use only `status = accepted` images as extra training photos, and compare a model trained with them against v1 on the **hand-boxed test set**. Keep them only if it is not worse.
3. Never put pseudo-labelled photos into validation or test, and never treat `no_detection` photos as healthy.